# Training ARIMA & BiLSTM — Peramalan Trafik Website Berita Online

**Skripsi:** Analisis Perbandingan ARIMA dan BiLSTM untuk Peramalan Trafik Website Berita Online Berbasis Dashboard

**Penyusun:** Salsabil Zahra (4123049) — STMIK Al Muslim Bekasi

---

Notebook ini melakukan:
1. Load dan eksplorasi dataset (CSV trafik website)
2. Preprocessing data (missing values, normalisasi, split data)
3. Pemodelan ARIMA (metodologi Box-Jenkins + auto_arima)
4. Pemodelan BiLSTM (TensorFlow/Keras)
5. Evaluasi (RMSE, MAE, MAPE)
6. Menyimpan seluruh hasil ke `results.json` dan `forecast_data.csv` untuk dashboard Streamlit

> **Cara pakai:** Upload file CSV trafik kamu ke Colab (format `Date, Visits` atau format wide Kaggle dengan kolom `Page` + kolom tanggal), lalu jalankan semua cell secara berurutan.

## 0. Instalasi Library (khusus Google Colab)

In [ ]:
# Jalankan cell ini hanya jika berada di Google Colab
# (di Colab, sebagian besar library berikut sudah terpasang, tapi pmdarima kadang perlu diinstal ulang)
!pip install -q pmdarima statsmodels scikit-learn tensorflow

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
import pmdarima as pm

from tensorflow.keras.models import Sequential, save_model
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print('Semua library berhasil diimpor.')

## 2. Upload dan Load Dataset

Upload file CSV trafik website kamu. Notebook ini mendukung dua format:

- **Format long**: kolom `Date` dan `Visits` (1 series saja)
- **Format wide ala Kaggle**: kolom `Page` + kolom-kolom tanggal sebagai header (`2017-01-01`, `2017-01-02`, ...)

Jika format wide, notebook akan otomatis memilih satu halaman (baris) dengan variasi pola tertinggi sebagai subjek penelitian (sesuai rekomendasi Tholib dkk., 2023).

In [ ]:
# Upload file CSV (khusus Google Colab)
from google.colab import files
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
print(f'File terupload: {csv_filename}')

In [ ]:
# Jika TIDAK menggunakan Colab, set nama file CSV secara manual di sini:
# csv_filename = 'traffic_data.csv'

raw_df = pd.read_csv(csv_filename)
print('Bentuk data:', raw_df.shape)
raw_df.head()

In [ ]:
# ── Deteksi format dan konversi menjadi format long (Date, Visits) ─────────

def load_as_long_format(df):
    """
    Mendeteksi apakah dataframe berformat long (Date, Visits)
    atau wide ala Kaggle (Page + kolom-kolom tanggal), lalu
    mengonversinya menjadi format long dengan kolom Date dan Visits.
    """
    cols_lower = [c.lower() for c in df.columns]

    # Format long: ada kolom 'date' dan ('visits' atau 'visit' atau numerik lainnya)
    if 'date' in cols_lower:
        date_col = df.columns[cols_lower.index('date')]
        # cari kolom numerik selain date
        value_candidates = [c for c in df.columns if c != date_col]
        # prioritaskan kolom bernama visits/visit
        value_col = None
        for c in value_candidates:
            if c.lower() in ('visits', 'visit', 'value', 'traffic'):
                value_col = c
                break
        if value_col is None:
            # ambil kolom numerik pertama
            numeric_cols = df[value_candidates].select_dtypes(include=[np.number]).columns
            value_col = numeric_cols[0] if len(numeric_cols) > 0 else value_candidates[0]

        long_df = df[[date_col, value_col]].copy()
        long_df.columns = ['Date', 'Visits']
        page_name = 'Single Series'

    # Format wide ala Kaggle: ada kolom 'Page' + banyak kolom tanggal
    elif 'page' in cols_lower:
        page_col = df.columns[cols_lower.index('page')]
        date_cols = [c for c in df.columns if c != page_col]

        # pilih halaman dengan variasi (std) tertinggi -> pola paling fluktuatif
        variances = df[date_cols].astype(float).std(axis=1, skipna=True)
        best_idx = variances.idxmax()
        page_name = df.loc[best_idx, page_col]

        series = df.loc[best_idx, date_cols].astype(float)
        long_df = pd.DataFrame({'Date': date_cols, 'Visits': series.values})

    else:
        raise ValueError('Format CSV tidak dikenali. Pastikan ada kolom "Date" atau "Page".')

    long_df['Date'] = pd.to_datetime(long_df['Date'])
    long_df = long_df.sort_values('Date').reset_index(drop=True)
    return long_df, page_name


df, page_name = load_as_long_format(raw_df)
print(f'Subjek data: {page_name}')
print(f'Jumlah hari: {len(df)}')
print(f'Periode: {df["Date"].min().date()} s.d. {df["Date"].max().date()}')
df.head()

## 3. Data Understanding — Eksplorasi Data

In [ ]:
# Statistik deskriptif & pengecekan missing values
print('--- Info Dataset ---')
print(df.info())
print()
print('--- Statistik Deskriptif ---')
print(df['Visits'].describe())
print()
print(f"Jumlah missing values: {df['Visits'].isna().sum()}")

In [ ]:
# Visualisasi plot time series
plt.figure(figsize=(14, 5))
plt.plot(df['Date'], df['Visits'], color='#028090', linewidth=1.2)
plt.title(f'Trafik Harian — {page_name}')
plt.xlabel('Tanggal')
plt.ylabel('Jumlah Pengunjung (Visits)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Data Preparation

Tahapan:
1. Penanganan missing values (interpolasi linier)
2. Set index waktu
3. Split data 80% training — 20% testing (berurutan sesuai waktu)
4. Normalisasi MinMaxScaler (khusus untuk BiLSTM)
5. Pembuatan sekuens input (lookback window) untuk BiLSTM

In [ ]:
# 1. Penanganan missing values dengan interpolasi linier
df['Visits'] = df['Visits'].interpolate(method='linear', limit_direction='both')
print(f"Missing values setelah interpolasi: {df['Visits'].isna().sum()}")

# 2. Set index waktu
df = df.set_index('Date')
ts = df['Visits'].astype(float)

# 3. Split 80% training - 20% testing
split_idx = int(len(ts) * 0.8)
train, test = ts.iloc[:split_idx], ts.iloc[split_idx:]
print(f'Data Training : {len(train)} hari')
print(f'Data Testing  : {len(test)} hari')

In [ ]:
# Uji Stasioneritas - Augmented Dickey-Fuller (ADF)
adf_result = adfuller(train)
print(f'ADF Statistic : {adf_result[0]:.4f}')
print(f'p-value       : {adf_result[1]:.4f}')
if adf_result[1] <= 0.05:
    print('=> Data stasioner (tolak H0)')
else:
    print('=> Data TIDAK stasioner, differencing diperlukan (gagal tolak H0)')

## 5. Pemodelan ARIMA

Mengikuti metodologi Box-Jenkins:
1. Identifikasi orde (p, d, q) menggunakan `auto_arima` berdasarkan kriteria AIC
2. Estimasi parameter (MLE — dilakukan otomatis oleh `pmdarima`)
3. Pemeriksaan diagnostik residual (uji Ljung-Box)
4. Peramalan pada data uji

In [ ]:
# 1 & 2. Identifikasi orde optimal dan estimasi parameter via auto_arima
arima_model = pm.auto_arima(
    train,
    start_p=0, start_q=0,
    max_p=5, max_q=5,
    d=None,                 # auto-detect differencing
    seasonal=False,
    stepwise=True,
    suppress_warnings=True,
    information_criterion='aic',
    trace=True
)

print(arima_model.summary())
best_order = arima_model.order
print(f'\nOrde terbaik ARIMA(p,d,q) = {best_order}')

In [ ]:
# 3. Pemeriksaan diagnostik residual - uji Ljung-Box
residuals = arima_model.resid()
lb_test = acorr_ljungbox(residuals, lags=[10], return_df=True)
print('Uji Ljung-Box (residual white noise check):')
print(lb_test)

if lb_test['lb_pvalue'].iloc[0] > 0.05:
    print('\n=> Residual bersifat white noise (model valid)')
else:
    print('\n=> Residual menunjukkan autokorelasi (model mungkin perlu disesuaikan)')

In [ ]:
# 4. Peramalan pada data uji
n_test = len(test)
arima_forecast, arima_conf_int = arima_model.predict(n_periods=n_test, return_conf_int=True)
arima_forecast = pd.Series(arima_forecast, index=test.index)

print('5 prediksi pertama ARIMA:')
print(arima_forecast.head())

## 6. Pemodelan BiLSTM

Arsitektur: `Bidirectional LSTM -> Dropout -> Dense`, optimizer Adam, loss MSE, Early Stopping.

Hyperparameter (sesuai kerangka pemikiran Bab II):
- `units`: jumlah neuron BiLSTM (64)
- `lookback`: panjang sekuens input (14 hari)
- `dropout`: 0.2
- `epochs`: maksimum 100 (dengan Early Stopping, patience=10)
- `batch_size`: 16

In [ ]:
# ── Hyperparameter ───────────────────────────────────────────────────────
LOOKBACK = 14
UNITS = 64
DROPOUT = 0.2
EPOCHS = 100
BATCH_SIZE = 16
PATIENCE = 10
LEARNING_RATE = 0.001

# ── Normalisasi MinMaxScaler ─────────────────────────────────────────────
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train.values.reshape(-1, 1))
test_scaled = scaler.transform(test.values.reshape(-1, 1))

# Gabungkan ekor data training (sebanyak lookback) ke data uji,
# agar sekuens pertama pada data uji dapat dibentuk
full_scaled = np.concatenate([train_scaled[-LOOKBACK:], test_scaled])

def create_sequences(data, lookback):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, LOOKBACK)
X_test, y_test = create_sequences(full_scaled, LOOKBACK)

# Reshape untuk input LSTM: (samples, timesteps, features)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print(f'X_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')

In [ ]:
# ── Arsitektur BiLSTM ────────────────────────────────────────────────────
bilstm_model = Sequential([
    Bidirectional(LSTM(UNITS, activation='tanh', return_sequences=False),
                   input_shape=(LOOKBACK, 1)),
    Dropout(DROPOUT),
    Dense(1)
])

bilstm_model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mse')
bilstm_model.summary()

In [ ]:
# ── Training dengan Early Stopping ──────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    restore_best_weights=True
)

history = bilstm_model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Plot training history (loss curve)
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curve — BiLSTM')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Prediksi pada data uji ──────────────────────────────────────────────
bilstm_pred_scaled = bilstm_model.predict(X_test)
bilstm_pred = scaler.inverse_transform(bilstm_pred_scaled).flatten()
bilstm_forecast = pd.Series(bilstm_pred, index=test.index)

print('5 prediksi pertama BiLSTM:')
print(bilstm_forecast.head())

## 7. Evaluasi Model — RMSE, MAE, MAPE

Persamaan:
- $RMSE = \\sqrt{\\frac{1}{n}\\sum (y_i - \\hat{y}_i)^2}$
- $MAE = \\frac{1}{n}\\sum |y_i - \\hat{y}_i|$
- $MAPE = \\frac{100\\%}{n}\\sum \\left|\\frac{y_i - \\hat{y}_i}{y_i}\\right|$

In [ ]:
def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

arima_metrics = evaluate(test.values, arima_forecast.values)
bilstm_metrics = evaluate(test.values, bilstm_forecast.values)

metrics_df = pd.DataFrame({
    'ARIMA': arima_metrics,
    'BiLSTM': bilstm_metrics
}).T

print(metrics_df.round(3))

best_model = metrics_df['RMSE'].idxmin()
print(f'\nModel dengan RMSE terkecil: {best_model}')

In [ ]:
# Visualisasi perbandingan: Aktual vs ARIMA vs BiLSTM
plt.figure(figsize=(14, 5))
plt.plot(test.index, test.values, label='Aktual', color='#1E293B', linewidth=2)
plt.plot(arima_forecast.index, arima_forecast.values, label='ARIMA', color='#F97316', linestyle='--')
plt.plot(bilstm_forecast.index, bilstm_forecast.values, label='BiLSTM', color='#02C39A', linestyle='--')
plt.title('Perbandingan Prediksi: Aktual vs ARIMA vs BiLSTM (Data Uji)')
plt.xlabel('Tanggal')
plt.ylabel('Jumlah Pengunjung (Visits)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Simpan Hasil untuk Dashboard Streamlit

Menyimpan 3 file:
1. `forecast_data.csv` — data historis + prediksi ARIMA & BiLSTM
2. `results.json` — metadata, metrik evaluasi, parameter model
3. `bilstm_model.h5` — model BiLSTM terlatih (opsional, untuk prediksi masa depan)

Setelah ini, download ketiga file dan letakkan di folder `outputs/` pada project dashboard Streamlit.

In [ ]:
# 1. forecast_data.csv — gabungkan seluruh data historis dengan hasil prediksi
full_result = pd.DataFrame({'Date': ts.index, 'Actual': ts.values})
full_result['ARIMA_Pred'] = np.nan
full_result['BiLSTM_Pred'] = np.nan

full_result.loc[full_result['Date'].isin(arima_forecast.index), 'ARIMA_Pred'] = arima_forecast.values
full_result.loc[full_result['Date'].isin(bilstm_forecast.index), 'BiLSTM_Pred'] = bilstm_forecast.values
full_result['DataSplit'] = ['Train'] * len(train) + ['Test'] * len(test)

full_result.to_csv('forecast_data.csv', index=False)
print('Tersimpan: forecast_data.csv')
full_result.tail()

In [ ]:
# 2. results.json — metadata dan metrik evaluasi
results = {
    'page_name': str(page_name),
    'n_total': int(len(ts)),
    'n_train': int(len(train)),
    'n_test': int(len(test)),
    'arima': {
        'order': list(best_order),
        'metrics': {k: float(v) for k, v in arima_metrics.items()},
        'ljung_box_pvalue': float(lb_test['lb_pvalue'].iloc[0])
    },
    'bilstm': {
        'units': UNITS,
        'lookback': LOOKBACK,
        'dropout': DROPOUT,
        'batch_size': BATCH_SIZE,
        'epochs_run': len(history.history['loss']),
        'metrics': {k: float(v) for k, v in bilstm_metrics.items()}
    },
    'best_model': best_model,
    'adf_pvalue': float(adf_result[1])
}

with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Tersimpan: results.json')
print(json.dumps(results, indent=2))

In [ ]:
# 3. Simpan model BiLSTM terlatih (opsional - untuk prediksi masa depan di dashboard)
bilstm_model.save('bilstm_model.h5')
print('Tersimpan: bilstm_model.h5')

# Simpan juga scaler agar konsisten saat prediksi masa depan
import joblib
joblib.dump(scaler, 'scaler.save')
print('Tersimpan: scaler.save')

In [ ]:
# Download ketiga file (khusus Google Colab)
from google.colab import files
files.download('forecast_data.csv')
files.download('results.json')
files.download('bilstm_model.h5')
files.download('scaler.save')

print('Selesai! Letakkan semua file ini di folder outputs/ pada project dashboard Streamlit.')

---
## Selesai

File yang dihasilkan:
- `forecast_data.csv`
- `results.json`
- `bilstm_model.h5`
- `scaler.save`

Pindahkan semua file ini ke folder `outputs/` pada project dashboard Streamlit, kemudian jalankan `streamlit run app.py`.